In [1]:
import pandas as pd

In [2]:
col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

In [3]:
train = pd.read_csv("../data/KDDTrain+.txt", header=None, names=col_names)
test = pd.read_csv("../data/KDDTest+.txt", header=None, names=col_names)

In [4]:
train["label"].value_counts()

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Grouping the category

In [5]:
label_category = {
    "Dos" : ['back', 'land', 'neptune', 'pod', 'smurf', 'teardrop'],
    "R2L" : ['ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy', 'warezclient', 'warezmaster'],
    "U2R" : ['buffer_overflow', 'loadmodule', 'perl', 'rootkit'],
    "Probe" : ['ipsweep', 'nmap', 'portsweep', 'satan']
}



In [6]:
# Flatten dictionary into one lookup: attack name -> category
attack_to_category = {}
for category, attacks in label_category.items():
    for attack in attacks:
        attack_to_category[attack] = category

# Verify: any real label not covered?
all_labels = set(train["label"].unique())
covered = set(attack_to_category.keys()) | {"normal"}
missing = all_labels - covered
print("Uncovered labels:", missing)

# Build the multi-class category column
def map_to_category(label):
    if label == "normal":
        return "normal"
    return attack_to_category.get(label, "unknown")

train["attack_category"] = train["label"].apply(map_to_category)
test["attack_category"] = test["label"].apply(map_to_category)

print(train["attack_category"].value_counts())


Uncovered labels: set()
attack_category
normal    67343
Dos       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [7]:
print(test["attack_category"].value_counts())

attack_category
normal     9711
Dos        5741
unknown    3750
R2L        2199
Probe      1106
U2R          37
Name: count, dtype: int64


In [8]:
# Verify: any real label not covered in test?
all_labels = set(test["label"].unique())
covered = set(attack_to_category.keys()) | {"normal"}
missing = all_labels - covered
print("Uncovered labels:", missing)

Uncovered labels: {'httptunnel', 'ps', 'xterm', 'saint', 'udpstorm', 'worm', 'mscan', 'named', 'sendmail', 'sqlattack', 'snmpguess', 'snmpgetattack', 'mailbomb', 'xsnoop', 'xlock', 'apache2', 'processtable'}


new label category as last category had some missing label from test file   

In [9]:
label_category_01 = {
    "Dos" : ['back', 'land', 'neptune', 'pod', 'smurf', 'teardrop', 'apache2', 'mailbomb', 'processtable','udpstorm', 'worm'],
    "R2L" : ['ftp_write', 'guess_passwd', 'httptunnel','imap', 'multihop', 'named', 'phf', 'sendmail', 'snmpgetattack', 'spy', 'snmpguess','warezclient', 'warezmaster','xlock', 'xsnoop'],
    "U2R" : ['buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'ps', 'sqlattack', 'xterm'],
    "Probe" : ['ipsweep', 'mscan', 'nmap', 'portsweep', 'saint','satan']
}

In [10]:
# Flatten dictionary into one lookup: attack name -> category
attack_to_category_01 = {}
for category, attacks in label_category_01.items():
    for attack in attacks:
        attack_to_category_01[attack] = category

# Verify: any real label not covered in test?
all_labels = set(test["label"].unique())
covered = set(attack_to_category_01.keys()) | {"normal"}
missing = all_labels - covered
print("Uncovered labels:", missing)

Uncovered labels: set()


rebuilding multi-class category column 


In [11]:
# Build the multi-class category column
def map_to_category_01(label):
    if label == "normal":
        return "normal"
    return attack_to_category_01.get(label, "unknown")

train["attack_category"] = train["label"].apply(map_to_category_01)
test["attack_category"] = test["label"].apply(map_to_category_01)

print(train["attack_category"].value_counts())

attack_category
normal    67343
Dos       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [12]:
print(test["attack_category"].value_counts())

attack_category
normal    9711
Dos       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64


In [13]:
test.head()


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,attack_category
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,Dos
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,Dos
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15,Probe
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11,Probe


Encoding columns


In [14]:
from sklearn.preprocessing import OneHotEncoder

categorical_clmns = ["protocol_type", "service", "flag"]
encoder = OneHotEncoder(sparse_output = False, handle_unknown= "ignore")
encoder.fit(train[categorical_clmns])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

Transforming train and test data with fitted encoder


In [15]:
train_encoded = encoder.transform(train[categorical_clmns])
test_encoded = encoder.transform(test[categorical_clmns])

print("Train encoded shape:", train_encoded.shape)
print("Test encoded shape:", test_encoded.shape)

Train encoded shape: (125973, 84)
Test encoded shape: (22544, 84)


In [16]:
encoded_clmn_names = encoder.get_feature_names_out(categorical_clmns)

train_encoded_df = pd.DataFrame(train_encoded, columns= encoded_clmn_names, index=train.index)
test_encoded_df = pd.DataFrame(test_encoded, columns= encoded_clmn_names, index=test.index)

train_encoded_df.head()

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Scaling Data


In [17]:
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,attack_category
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,Dos
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,normal


In [18]:
excluded_clmns = categorical_clmns + ["label", "difficulty", "attack_category"]
numeric_clmns = [col for col in train.columns if col not in excluded_clmns]

print(numeric_clmns)

['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


Extracted those column which all are numerics 


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train[numeric_clmns])

StandardScaler()

In [20]:
train_scaled = scaler.transform(train[numeric_clmns])
test_scaled = scaler.transform(test[numeric_clmns])

print("Train encoded shape:", train_scaled.shape)
print("Test encoded shape:", test_scaled.shape)

Train encoded shape: (125973, 38)
Test encoded shape: (22544, 38)


In [21]:
train_scaled_df = pd.DataFrame(train_scaled, columns= numeric_clmns, index=train.index)
test_scaled_df = pd.DataFrame(test_scaled, columns= numeric_clmns, index=test.index)

train_scaled_df.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,-0.110249,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,-0.324063,-0.818890,-0.782367,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387
1,-0.110249,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,0.734343,-1.035688,-1.161030,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387
2,-0.110249,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,0.734343,-0.809857,-0.938287,-0.174417,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387
3,-0.110249,-0.007723,-0.002891,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,1.235694,-0.011664,...,-1.533670,1.258754,1.066401,-0.439078,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084
4,-0.110249,-0.007728,-0.004814,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,1.235694,-0.011664,...,0.734343,1.258754,1.066401,-0.439078,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387


Merging all scaled and encoded data

In [22]:
train_final = pd.concat([train_encoded_df, train_scaled_df, train["attack_category"]], axis =1)
test_final = pd.concat([test_encoded_df, test_scaled_df, test["attack_category"]], axis =1)

print("Train final shape", train_final.shape)
print("Test final shape", test_final.shape)

train_final.head()

Train final shape (125973, 123)
Test final shape (22544, 123)


,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_category
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.818890,-0.782367,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387,normal
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.035688,-1.161030,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,normal
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.809857,-0.938287,-0.174417,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387,Dos
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.258754,1.066401,-0.439078,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084,normal
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.258754,1.066401,-0.439078,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,normal


saving the dataset as new data file in data folder

In [23]:
train_final.to_csv("../data/processed_train_multiclass.csv", index=False)
test_final.to_csv("../data/processed_test_multiclass.csv", index=False)

print("Saved processed_train_multiclass.csv and processed_test_multiclass.csv")

Saved processed_train_multiclass.csv and processed_test_multiclass.csv
